### Setup and imports

In [3]:
import pandas as pd
import numpy as np
import rdkit
import ast
import matplotlib.pyplot as plt
#add future imports here

In [4]:
df = pd.read_csv(r"C:\CSC 4444\project\dataset\SMILES_IR_Spectroscopy.csv", low_memory=False)

In [5]:
def plot_graph(row):
    x_coords = row["x_coords"]
    y_coords = row["y_coords"]

    if isinstance(x_coords, str):
        x_coords = ast.literal_eval(x_coords)
    if isinstance(y_coords, str):
        y_coords = ast.literal_eval(y_coords)

    plt.figure(figsize=(10, 4))
    plt.plot(x_coords, y_coords, linewidth=1)
    plt.xlabel("1/cm")
    plt.ylabel("Absorbance")
    plt.title(row.get("smiles", "IR Spectrum"))
    plt.tight_layout()
    plt.show()
    return None

In [ ]:
#test it
plot_graph(df.iloc[0])
for i in range(15):
    plot_graph(df.iloc[i])

## Normalization of data

### Standardize spectra through interpolating coordinate data to fixed grid 
* Using indiviail based normalizaiton and not group global normalization because:
  * experimental conditions may not have been consistant
  * the model learns scale artifacts (instrument, concentration) which makes it worse at generalization

In [ ]:
# Perform row-wise min-max normalization on absorbance values.
def min_max_normalize_y_coords(y_coords):
    if isinstance(y_coords, str):
        y_coords = ast.literal_eval(y_coords)

    y_values = np.asarray(y_coords, dtype=float)
    if y_values.size == 0:
        return [], np.nan, np.nan, np.nan

    y_min = float(np.min(y_values))
    y_max = float(np.max(y_values))
    y_range = y_max - y_min

    if y_range == 0:
        normalized_y = np.zeros_like(y_values, dtype=float)
    else:
        normalized_y = (y_values - y_min) / y_range

    return normalized_y.tolist(), y_min, y_max, y_range


normalized_y_df = df.copy()
y_normalization_metadata = normalized_y_df["y_coords"].apply(min_max_normalize_y_coords)

normalized_y_df["y_coords"] = y_normalization_metadata.apply(lambda values: values[0]).astype(object)
normalized_y_df["y_min_before_minmax"] = y_normalization_metadata.apply(lambda values: values[1])
normalized_y_df["y_max_before_minmax"] = y_normalization_metadata.apply(lambda values: values[2])
normalized_y_df["y_range_before_minmax"] = y_normalization_metadata.apply(lambda values: values[3])
normalized_y_df["y_normalization"] = "row_min_max_0_1"


In [ ]:
#test it
plot_graph(normalized_y_df.iloc[0])
for i in range(15):
    print(f'===================')
    print(f'====regular{i}=====')
    plot_graph(df.iloc[i])
    print(f'====normalized{i}=====')
    plot_graph(normalized_y_df.iloc[i])
    print(f'===================')

## Labeling

### Canonicallize smiles 
* learn more at 
  * https://luis-vollmers.medium.com/tutorial-to-smiles-and-canonical-smiles-explained-with-examples-fbc8a46ca29f
  * could be done via rdkit

### Create Labels 

### Generate fingerprint targets

## Data Splitting

### Train / Validation / Test Split

## Baseline Modeling

### Build Simple 1D CNN Baseline
* subjet to change

### Compare Against Classical Baseline

## Main Model

### Build CNN with Multi-Task Outputs

### Train the Model

## Evaluation
* for results section

### Evaluate Functional Group Prediction (F1, Precision, Recall)

### Evaluate Retrieval Performance (Top-K Accuracy, MRR)

### Calibration and Confidence Analysis

## Deployment Design

### Functional Group Probability Outputs

### Top-K SMILES Candidate Retrieval

### Confidence and Uncertainty Display

In [13]:
#Smiles canonicalization implementation
from rdkit import Chem

def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol)
    except:
        return None

df['canonical_smiles'] = df['smiles'].apply(canonicalize_smiles)

print(f"Total rows: {len(df)}")
print(f"Successfully canonicalized: {df['canonical_smiles'].notna().sum()}")

df.to_csv(r"C:\CSC 4444\project\dataset\SMILES_IR_Spectroscopy.csv", index=False)
print("Dataset updated.")

print(df.columns.tolist())
df[['smiles', 'canonical_smiles']]

[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:37] WARNING: not removing hydrogen atom without neighbors
[20:03:38] WARNING: not removing hydrogen atom without neighbors
[20:03:38] WARNING: not removing hydrogen atom without neighbors
[20:03:38] WARNING: not removing hydrogen atom without neighbors
[20:03:38] WARNING: not removing hydrogen atom without neighbors
[20:03:39] WARNING: not removing hydrogen atom without neighbors
[20:03:39] WARNING: not removing hydrogen atom without neighbors
[20:03:39] WARNING: not removing hydrogen atom without neighbors
[20:03:39] WARNING: not removing hydrogen atom without neighbors
[20:03:39] WARNING: not r

Total rows: 8994
Successfully canonicalized: 8994
Dataset updated.
['filename', 'smiles', 'origin_xunits', 'origin_yunits', 'x_coords', 'y_coords', 'canonical_smiles']


,smiles,canonical_smiles
0,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,CC(=O)N[C@H]1[C@H]([C@H](O)[C@H](O)CO)O[C@](O)...
1,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,CC(=O)N[C@H]1[C@H]([C@H](O)[C@H](O)CO)O[C@](O)...
2,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,CC(=O)N[C@H]1[C@H]([C@H](O)[C@H](O)CO)O[C@](O)...
3,[O-][N+](=O)c1ccc(Cl)cc1,O=[N+]([O-])c1ccc(Cl)cc1
4,Nc1ccc(cc1)[N+]([O-])=O,Nc1ccc([N+](=O)[O-])cc1
...,...,...
8989,CCCCCC(Br)CC,CCCCCC(Br)CC
8990,OC(=O)c1ccc(O)cc1,O=C(O)c1ccc(O)cc1
8991,CC#CC(C)(C)C,CC#CC(C)(C)C
8992,CN(C)c1ccc(C)cc1,Cc1ccc(N(C)C)cc1
